# DNAKit 高级端到端工作流

本 Notebook 使用固定本地序列连接模式、热力学、sketch、聚类/泄漏、参考库评价、分子生物学和自包含报告。它不访问网络；NUPACK 未由项目安装或尝试安装，未探测、导入或调用。文件只写入临时目录。

In [ ]:
import tempfile
from io import StringIO
from pathlib import Path

import dnakit
from dnakit import DNASequence, DNASet, read_set
from dnakit.datasets import (
    ClusterConfig,
    HierarchicalClusteringConfig,
    LeakageConfig,
    SplitConfig,
    cluster_sequences,
    detect_leakage,
    hierarchical_cluster,
    select_representatives,
    split,
)
from dnakit.evaluation import (
    ReferenceSearchConfig,
    create_reference_library,
    evaluate_memorization,
    evaluate_novelty,
    evaluate_synthesis_risk,
)
from dnakit.fingerprints import (
    minhash,
    multiscale_fingerprint,
    thermodynamic_fingerprint,
)
from dnakit.molbio import (
    Primer3CLIDesignAdapter,
    digest_restriction,
    generate_mutation_library,
    prepare_primer_design,
    primer_properties,
)
from dnakit.patterns import scan_motif, scan_orfs, scan_restriction_sites
from dnakit.similarity import build_sketch_index, nearest_neighbors
from dnakit.standardize import GapNormalizationConfig, normalize_gaps
from dnakit.thermodynamics import (
    Primer3CLIAdapter,
    ThermodynamicConditions,
    melting_temperature,
)
from dnakit.visualization import (
    ImageExportConfig,
    build_html_report,
    plot_sequence,
    save_html_report,
    save_image,
    save_svg,
)

assert dnakit.__version__ == "0.1.1"

## 1. 固定输入、模式和热力学

固定夹具含两条 exact duplicate 和一条单碱基差异。热力学显式记录条件与参数版本；可选 Primer3 分支只在本地后端可用且调用方显式提供 adapter 时执行。

In [ ]:
records = read_set(
    StringIO(
        ">seq-a fixed example\nACGTACGT\n"
        ">seq-b exact duplicate\nACGTACGT\n"
        ">seq-c one substitution\nACGTTCGT\n"
    ),
    format="fasta",
)
assert records.ids == ("seq-a", "seq-b", "seq-c")

gap_result = normalize_gaps(
    DNASequence("ACNNNNTG", alphabet="iupac"),
    config=GapNormalizationConfig(min_run_length=4),
)
motifs = scan_motif(records[0], "ACG", mode="exact", strand="forward")
orfs = scan_orfs(records[0], require_complete=False)
restriction_sites = scan_restriction_sites(DNASequence("TTTGAATTCAAA"), ("EcoRI",))
conditions = ThermodynamicConditions(sodium_molar=0.05, strand_concentration_molar=250e-9)
tm = melting_temperature(records[0].sequence, method="nearest_neighbor", conditions=conditions)
primer3_structure_adapter = Primer3CLIAdapter()
resolved_structure_adapter = (
    primer3_structure_adapter if primer3_structure_adapter.info.available else None
)

assert len(motifs.hits) == 2
assert gap_result.sequence.is_gapped
assert len(restriction_sites.hits) == 1
assert tm.conditions.sodium_molar == 0.05
assert tm.algorithm_version == "dnakit-santalucia1998-v1"

## 2. Sketch、Top-k 和版本化综合指纹

索引使用固定 seed 和固定 bottom-k schema；Top-k 是对内存 sketch 的确定性 exact scan。热力学指纹固定为 16 维 v2 schema，并显式记录结构特征是否可用。

In [ ]:
sketch = minhash(records[0], k=3, num_hashes=16, seed=7)
multiscale = multiscale_fingerprint(records[0], k_values=(1, 2))
thermo_fp = thermodynamic_fingerprint(
    records[0],
    structure_adapter=resolved_structure_adapter,
    paired_value=records[2],
)
sketch_index = build_sketch_index(records, k=3, num_hashes=16, seed=7)
neighbors = nearest_neighbors(records[0], sketch_index, top_k=3)

assert sketch.seed == 7
assert multiscale.feature_names[-1] == "global_gc"
assert thermo_fp.schema_version == "dnakit.thermodynamic_fingerprint.v2"
assert len(thermo_fp.values) == 16
assert thermo_fp.values[4] == float(resolved_structure_adapter is not None)
assert neighbors.hits[0].record_id == "seq-a"
assert neighbors.hits[0].similarity == 1.0

## 3. 聚类、划分、泄漏与版本化参考库

novelty/memorization 只相对于这个固定、带 digest 的本地参考库定义。

In [ ]:
cluster_config = ClusterConfig(method="identity", threshold=0.8, representative_policy="medoid")
clusters = cluster_sequences(records, config=cluster_config)
hierarchy = hierarchical_cluster(
    records,
    config=HierarchicalClusteringConfig(method="identity", linkage="average"),
)
representatives = select_representatives(records, clusters.labels, policy="medoid")
partitions = split(
    records,
    config=SplitConfig(method="random", ratios={"train": 2 / 3, "test": 1 / 3}, seed=11),
)
split_sets = {subset.name: subset.records for subset in partitions.subsets}
leakage = detect_leakage(split_sets, config=LeakageConfig(method="identity", threshold=0.9))

reference = create_reference_library(
    DNASet.from_records([records[0]]),
    name="fixed-notebook-reference",
    version="1",
    source="examples/fixed_demo.fasta",
    filters={"record_ids": ["seq-a"]},
)
reference_config = ReferenceSearchConfig(method="identity", copy_threshold=0.9)
novelty = evaluate_novelty(records, reference, config=reference_config)
memorization = evaluate_memorization(records, reference, config=reference_config)
risk = evaluate_synthesis_risk(records)

assert len(clusters.labels) == 3
assert len(hierarchy.linkage) == len(records) - 1
assert representatives.cluster_count == len(clusters.clusters)
assert sum(partitions.counts.values()) == 3
assert reference.digest_scope.startswith("ordered IDs")
assert novelty.metrics["query_count"] == 3
assert memorization.metrics["generated_count"] == 3
assert risk.metrics["record_count"] == 3

## 4. 分子生物学的序列级结果

酶切和突变文库只描述确定性序列，不预测实验产率。引物结构与设计同样只在调用方显式提供或执行 Primer3 adapter 时计算。

In [ ]:
digest = digest_restriction(DNASequence("TTTGAATTCAAA"), ("EcoRI",))
library = generate_mutation_library(
    DNASequence("ACGT"),
    {1: ("A", "G")},
    mode="single",
    seed=13,
)
properties = primer_properties(
    records[0].sequence,
    paired_primer=records[2].sequence,
    structure_adapter=resolved_structure_adapter,
)
design_request = prepare_primer_design(
    DNASequence("ACGT" * 100),
    target_start=150,
    target_end=200,
    primer_length_range=(18, 22),
    product_length_range=(100, 250),
    candidate_count=2,
)
design_adapter = Primer3CLIDesignAdapter()
designed = (
    design_adapter.design(design_request)
    if design_adapter.info.available
    else design_request
)

assert len(digest.cuts) == 1
assert len(digest.fragments) == 2
assert library.seed == 13
assert len(library.variants) == 2
assert properties.hairpin.available == (resolved_structure_adapter is not None)
assert designed.execution_performed == design_adapter.info.available

## 5. 自包含报告和多格式图片

所有输出写入临时目录；HTML 不加载网络资源，PNG 使用 `viz` extra。

In [ ]:
sequence_svg = plot_sequence(records[2])
html_report = build_html_report(
    records,
    results={"novelty": novelty, "memorization": memorization, "risk": risk},
    title="DNAKit fixed advanced workflow",
)

with tempfile.TemporaryDirectory() as directory:
    output = Path(directory)
    svg_saved = save_svg(sequence_svg, output / "sequence.svg")
    png_saved = save_image(
        sequence_svg,
        output / "sequence.png",
        config=ImageExportConfig(dpi=600, width=640),
    )
    html_saved = save_html_report(html_report, output / "report.html")
    assert svg_saved.target_artifact.byte_size > 0
    assert png_saved.format == "png"
    assert html_saved.target_artifact.byte_size > 0

summary = {
    "records": len(records),
    "clusters": len(clusters.clusters),
    "leakage_events": len(leakage.events),
    "reference_digest": reference.digest,
    "thermodynamic_fingerprint_schema": thermo_fp.schema_version,
    "primer3_available": resolved_structure_adapter is not None,
    "primer3_design_executed": designed.execution_performed,
    "nupack_used": False,
}
summary

## 结论与边界

该流程保留固定 seed、参考库 digest、算法版本、条件和资源上限。Primer3 为独立可选 adapter；NUPACK 未由项目安装或尝试安装，未探测、导入或调用。synthesis-risk 和分子生物学模拟不代表实验成功。